# 🏋️ Member 1 - Squat Pose Dataset Preprocessing Pipeline

> **Mục tiêu:** Preprocessing dataset squat cho YOLOv8-Pose
> 
> **Input:** Raw images + CSV annotations  
> **Output:** YOLO format dataset (train/val/test) ready cho training

---

## 📋 Pipeline Overview

1. **Setup Environment** - Mount Drive, install dependencies
2. **Filter CSV** - Chỉ giữ images có trên local
3. **Convert to YOLO** - Chuyển CSV annotations → YOLO pose format
4. **Resize Images** - Resize về 640×640 (YOLO standard)
5. **Split Dataset** - Chia train/val/test (70/15/15)
6. **Create data.yaml** - Config file cho YOLO
7. **Verify & Visualize** - Kiểm tra và vẽ samples

---

**Created:** 2026-04-02  
**Phase:** 2 (Preprocessing)  
**Member:** Data Engineer (CV)

## 1️⃣ Setup Environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Setup project paths
import os
from pathlib import Path

# Change these paths to match your Google Drive structure
PROJECT_ROOT = Path('/content/drive/MyDrive/Project_DeepLearning')
DATA_ROOT = PROJECT_ROOT / 'GYM data'
IMAGES_ROOT = DATA_ROOT / 'squat-20260402T134558Z-3-001' / 'squat'
CSV_PATH = DATA_ROOT / 'squat_local_filtered.csv'
OUTPUT_ROOT = DATA_ROOT / 'cv_dataset'

# Create output directories
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("✅ Paths configured:")
print(f"  Project: {PROJECT_ROOT}")
print(f"  Data: {DATA_ROOT}")
print(f"  Images: {IMAGES_ROOT}")
print(f"  CSV: {CSV_PATH}")
print(f"  Output: {OUTPUT_ROOT}")

In [ ]:
# Install required packages
!pip install -q opencv-python-headless tqdm pyyaml

In [ ]:
# Import libraries
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import shutil
import random
from tqdm.auto import tqdm
from pathlib import Path
import json

print("✅ All libraries imported successfully")

## 2️⃣ Filter CSV by Existing Images

Chỉ giữ lại các rows trong CSV có ảnh tồn tại trên local.

In [ ]:
def filter_csv_by_images(csv_path, images_root, output_csv):
    """
    Filter CSV to keep only rows where image exists in images_root
    """
    csv_path = Path(csv_path)
    images_root = Path(images_root)
    output_csv = Path(output_csv)
    
    # Read all images in local folder
    local_images = set()
    for img_path in images_root.glob("**/*.jpg"):
        local_images.add(img_path.name)
    
    print(f"📁 Found {len(local_images)} local images")
    
    # Read CSV
    df = pd.read_csv(csv_path)
    print(f"📊 CSV has {len(df)} total rows")
    
    # Get image column name
    img_col = 'image' if 'image' in df.columns else 'filename'
    
    # Filter rows
    df_filtered = df[df[img_col].isin(local_images)]
    
    skipped = len(df) - len(df_filtered)
    print(f"\n✅ Filtered: {len(df_filtered)} rows kept, {skipped} rows skipped")
    
    # Save filtered CSV
    df_filtered.to_csv(output_csv, index=False)
    
    print(f"💾 Saved filtered CSV: {output_csv}")
    print(f"\n📊 Summary:")
    print(f"  Total images in local: {len(local_images)}")
    print(f"  Matched in CSV: {len(df_filtered)}")
    print(f"  Coverage: {len(df_filtered)/len(local_images)*100:.1f}%")
    
    return df_filtered

# Run filter if CSV doesn't exist or needs update
if not CSV_PATH.exists() or input("Re-filter CSV? (y/n): ").lower() == 'y':
    df_filtered = filter_csv_by_images(
        csv_path=DATA_ROOT / 'yolo_keypoints_cleaned_final.csv',  # Original CSV
        images_root=IMAGES_ROOT,
        output_csv=CSV_PATH
    )
else:
    df_filtered = pd.read_csv(CSV_PATH)
    print(f"✅ Loaded existing filtered CSV: {len(df_filtered)} rows")

In [ ]:
# Preview CSV structure
print("\n📋 CSV Preview:")
print(df_filtered.head())
print(f"\n📐 Columns ({len(df_filtered.columns)}): {list(df_filtered.columns[:10])}...")

## 3️⃣ Convert CSV to YOLO Pose Format

Chuyển đổi CSV annotations thành YOLO format:
- Format: `class_id x_center y_center width height x1 y1 v1 x2 y2 v2 ... x17 y17 v17`
- Tất cả giá trị đều normalized [0, 1]

In [ ]:
def convert_csv_to_yolo_pose(csv_path, images_root, out_labels_dir, class_map={'squat': 0}):
    """
    Convert CSV with keypoints to YOLO pose format
    
    CSV format: image,label,x0,y0,x1,y1,...,x16,y16
    YOLO format: class x_c y_c w h x1 y1 v1 ... x17 y17 v17
    """
    csv_path = Path(csv_path)
    images_root = Path(images_root)
    out_labels_dir = Path(out_labels_dir)
    out_labels_dir.mkdir(parents=True, exist_ok=True)
    
    # Read CSV
    df = pd.read_csv(csv_path)
    img_col = 'image' if 'image' in df.columns else 'filename'
    
    success_count = 0
    error_count = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Converting to YOLO"):
        try:
            img_name = row[img_col]
            
            # Find image path
            img_path = images_root / img_name
            if not img_path.exists():
                possible = list(images_root.glob(f"**/{img_name}"))
                if possible:
                    img_path = possible[0]
                else:
                    error_count += 1
                    continue
            
            # Read image to get dimensions
            img = cv2.imread(str(img_path))
            if img is None:
                error_count += 1
                continue
            h, w = img.shape[:2]
            
            # Get class ID
            label_name = row.get('label', 'squat')
            class_id = class_map.get(label_name, 0)
            
            # Collect keypoints (17 points)
            kps = []
            for i in range(17):
                x_key = f'x{i}'
                y_key = f'y{i}'
                
                if x_key in row and y_key in row:
                    x_val = row[x_key]
                    y_val = row[y_key]
                    
                    if pd.notna(x_val) and pd.notna(y_val) and x_val != '' and y_val != '':
                        kps.append((float(x_val), float(y_val), 2))  # visible=2
                    else:
                        kps.append((None, None, 0))  # not visible
                else:
                    kps.append((None, None, 0))
            
            # Skip if no keypoints
            if all(k[0] is None for k in kps):
                error_count += 1
                continue
            
            # Compute bounding box from visible keypoints
            xs = [k[0] for k in kps if k[0] is not None]
            ys = [k[1] for k in kps if k[1] is not None]
            
            if not xs or not ys:
                error_count += 1
                continue
            
            xmin, xmax = min(xs), max(xs)
            ymin, ymax = min(ys), max(ys)
            
            # Add 5% padding
            pad_x = (xmax - xmin) * 0.05
            pad_y = (ymax - ymin) * 0.05
            xmin = max(0, xmin - pad_x)
            ymin = max(0, ymin - pad_y)
            xmax = min(w, xmax + pad_x)
            ymax = min(h, ymax + pad_y)
            
            # Calculate normalized bbox
            box_w = xmax - xmin
            box_h = ymax - ymin
            x_center = (xmin + box_w / 2) / w
            y_center = (ymin + box_h / 2) / h
            box_w_n = box_w / w
            box_h_n = box_h / h
            
            # Normalize keypoints
            kp_parts = []
            for (xk, yk, vis) in kps:
                if xk is None or yk is None:
                    kp_parts.extend(["0", "0", "0"])
                else:
                    kp_parts.extend([f"{(xk/w):.6f}", f"{(yk/h):.6f}", str(int(vis))])
            
            # Compose YOLO label line
            line_parts = [
                str(class_id),
                f"{x_center:.6f}",
                f"{y_center:.6f}",
                f"{box_w_n:.6f}",
                f"{box_h_n:.6f}"
            ] + kp_parts
            line = ' '.join(line_parts)
            
            # Save label file
            out_label_path = out_labels_dir / f"{img_path.stem}.txt"
            with open(out_label_path, 'w') as f:
                f.write(line + '\n')
            
            success_count += 1
            
        except Exception as e:
            error_count += 1
            continue
    
    print(f"\n✅ Conversion complete!")
    print(f"  Success: {success_count}")
    print(f"  Errors: {error_count}")
    print(f"  Output: {out_labels_dir}")
    
    return success_count

# Run conversion
labels_dir = OUTPUT_ROOT / 'all_labels'
n_labels = convert_csv_to_yolo_pose(
    csv_path=CSV_PATH,
    images_root=IMAGES_ROOT,
    out_labels_dir=labels_dir,
    class_map={'squat': 0}
)

## 4️⃣ Resize Images to 640×640

YOLO training yêu cầu input size cố định. Standard size là 640×640.

In [ ]:
def resize_images(input_dir, output_dir, target_size=640):
    """
    Resize all images to target_size x target_size
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Find all images
    all_images = list(input_dir.glob("**/*.jpg")) + list(input_dir.glob("**/*.png"))
    print(f"📁 Found {len(all_images)} images to resize")
    
    success_count = 0
    error_count = 0
    
    for img_path in tqdm(all_images, desc="Resizing images"):
        try:
            # Read image
            img = cv2.imread(str(img_path))
            if img is None:
                error_count += 1
                continue
            
            # Resize
            resized = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_LINEAR)
            
            # Save (flat structure with original name)
            output_path = output_dir / img_path.name
            cv2.imwrite(str(output_path), resized, [cv2.IMWRITE_JPEG_QUALITY, 95])
            
            success_count += 1
            
        except Exception as e:
            error_count += 1
            continue
    
    print(f"\n✅ Resize complete!")
    print(f"  Success: {success_count}")
    print(f"  Errors: {error_count}")
    print(f"  Output: {output_dir}")
    
    return success_count

# Run resize
resized_dir = OUTPUT_ROOT / 'images_resized'
n_resized = resize_images(
    input_dir=IMAGES_ROOT,
    output_dir=resized_dir,
    target_size=640
)

# Verify first image size
first_img = list(resized_dir.glob("*.jpg"))[0]
img_check = cv2.imread(str(first_img))
print(f"\n🔍 Verification - First image shape: {img_check.shape}")

## 5️⃣ Split Dataset (Train/Val/Test)

Chia dataset theo tỷ lệ:
- **Train:** 70% (dùng để học)
- **Val:** 15% (dùng để tune hyperparameters)
- **Test:** 15% (đánh giá cuối cùng)

In [ ]:
def split_yolo_dataset(images_dir, labels_dir, output_dir, train_ratio=0.7, val_ratio=0.15, seed=42):
    """
    Split dataset into train/val/test
    """
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)
    output_dir = Path(output_dir)
    
    # Find all images
    all_images = list(images_dir.glob("*.jpg"))
    
    # Match with labels
    pairs = []
    for img_path in all_images:
        label_path = labels_dir / f"{img_path.stem}.txt"
        if label_path.exists():
            pairs.append((img_path, label_path))
    
    print(f"📊 Found {len(pairs)} valid image-label pairs")
    
    if len(pairs) == 0:
        print("❌ ERROR: No valid pairs found!")
        return
    
    # Shuffle with seed for reproducibility
    random.seed(seed)
    random.shuffle(pairs)
    
    # Calculate split sizes
    n_total = len(pairs)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)
    
    splits = {
        'train': pairs[:n_train],
        'val': pairs[n_train:n_train+n_val],
        'test': pairs[n_train+n_val:]
    }
    
    print(f"\n📐 Split sizes:")
    for split_name, split_pairs in splits.items():
        pct = len(split_pairs) / n_total * 100
        print(f"  {split_name:5s}: {len(split_pairs):4d} ({pct:.1f}%)")
    
    # Create directories
    for split_name in ['train', 'val', 'test']:
        (output_dir / split_name / 'images').mkdir(parents=True, exist_ok=True)
        (output_dir / split_name / 'labels').mkdir(parents=True, exist_ok=True)
    
    # Copy files
    for split_name, split_pairs in splits.items():
        for img_path, label_path in tqdm(split_pairs, desc=f"Copying {split_name}"):
            # Copy image
            shutil.copy(img_path, output_dir / split_name / 'images' / img_path.name)
            # Copy label
            shutil.copy(label_path, output_dir / split_name / 'labels' / label_path.name)
    
    print(f"\n✅ Dataset split complete!")
    print(f"  Output directory: {output_dir}")
    
    return splits

# Run split
splits = split_yolo_dataset(
    images_dir=resized_dir,
    labels_dir=labels_dir,
    output_dir=OUTPUT_ROOT,
    train_ratio=0.7,
    val_ratio=0.15,
    seed=42
)

In [ ]:
# Verify split structure
print("\n📁 Dataset Structure:")
for split in ['train', 'val', 'test']:
    img_count = len(list((OUTPUT_ROOT / split / 'images').glob('*.jpg')))
    lbl_count = len(list((OUTPUT_ROOT / split / 'labels').glob('*.txt')))
    print(f"  {split:5s} => Images: {img_count:4d}, Labels: {lbl_count:4d}")

## 6️⃣ Create data.yaml

YOLOv8-Pose cần file config để biết:
- Dataset paths
- Class names
- Keypoint configuration

In [ ]:
# Create data.yaml config
data_yaml = {
    'path': '.',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    
    'nc': 1,
    'names': ['squat'],
    
    'kpt_shape': [17, 3],  # [num_keypoints, (x, y, visibility)]
    
    'names_kpt': [
        'nose',           # 0
        'left_eye',       # 1
        'right_eye',      # 2
        'left_ear',       # 3
        'right_ear',      # 4
        'left_shoulder',  # 5
        'right_shoulder', # 6
        'left_elbow',     # 7
        'right_elbow',    # 8
        'left_wrist',     # 9
        'right_wrist',    # 10
        'left_hip',       # 11
        'right_hip',      # 12
        'left_knee',      # 13
        'right_knee',     # 14
        'left_ankle',     # 15
        'right_ankle'     # 16
    ],
    
    'flip_idx': [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15],
    
    'info': {
        'description': 'Squat pose estimation dataset',
        'version': '1.0',
        'year': 2026,
        'total_images': n_resized,
        'split': {
            'train': len(splits['train']) if splits else 0,
            'val': len(splits['val']) if splits else 0,
            'test': len(splits['test']) if splits else 0
        }
    }
}

# Save data.yaml
yaml_path = OUTPUT_ROOT / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print("✅ data.yaml created successfully!")
print(f"  Location: {yaml_path}")
print(f"\n📄 Content preview:")
print(yaml.dump(data_yaml, default_flow_style=False, sort_keys=False))

## 7️⃣ Verify & Visualize Dataset

Kiểm tra dataset integrity và visualize một số samples.

In [ ]:
def draw_keypoints_on_image(image, label_path):
    """
    Draw keypoints and skeleton on image
    """
    # Read label
    with open(label_path, 'r') as f:
        line = f.readline().strip()
    
    parts = line.split()
    if len(parts) < 56:  # 5 (bbox) + 17*3 (kpts) = 56
        return image
    
    # Parse bbox (normalized)
    class_id = int(parts[0])
    x_center, y_center, width, height = map(float, parts[1:5])
    
    h, w = image.shape[:2]
    
    # Draw bbox (unnormalize)
    x1 = int((x_center - width/2) * w)
    y1 = int((y_center - height/2) * h)
    x2 = int((x_center + width/2) * w)
    y2 = int((y_center + height/2) * h)
    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
    
    # Draw keypoints
    kpts = []
    for i in range(17):
        kp_x = float(parts[5 + i*3])
        kp_y = float(parts[5 + i*3 + 1])
        kp_vis = int(parts[5 + i*3 + 2])
        
        if kp_vis > 0 and kp_x > 0 and kp_y > 0:  # Visible
            px = int(kp_x * w)
            py = int(kp_y * h)
            kpts.append((px, py))
            
            # Draw circle
            cv2.circle(image, (px, py), 4, (0, 0, 255), -1)
            # Draw keypoint index
            cv2.putText(image, str(i), (px+5, py-5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
        else:
            kpts.append(None)
    
    # Draw skeleton connections
    connections = [
        (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),  # Arms
        (5, 11), (6, 12), (11, 12),                # Torso
        (11, 13), (13, 15), (12, 14), (14, 16)    # Legs
    ]
    
    for pt1_idx, pt2_idx in connections:
        if kpts[pt1_idx] is not None and kpts[pt2_idx] is not None:
            cv2.line(image, kpts[pt1_idx], kpts[pt2_idx], (255, 0, 0), 2)
    
    return image

In [ ]:
# Visualize random samples
n_samples = 9
train_images = list((OUTPUT_ROOT / 'train' / 'images').glob("*.jpg"))
samples = random.sample(train_images, min(n_samples, len(train_images)))

# Create visualization grid
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

for i, img_path in enumerate(samples):
    label_path = OUTPUT_ROOT / 'train' / 'labels' / f"{img_path.stem}.txt"
    
    if not label_path.exists():
        continue
    
    # Read and draw
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_vis = draw_keypoints_on_image(img.copy(), label_path)
    
    # Plot
    axes[i].imshow(img_vis)
    axes[i].set_title(f"Sample {i+1}")
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'visualization_samples.jpg', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Visualization complete!")
print(f"  Saved: {OUTPUT_ROOT / 'visualization_samples.jpg'}")

## 8️⃣ Generate Dataset Statistics

In [ ]:
def generate_statistics(dataset_root):
    """
    Generate dataset statistics
    """
    dataset_root = Path(dataset_root)
    
    stats = {
        "dataset_name": "Squat Pose Estimation",
        "version": "1.0",
        "created_date": "2026-04-02",
        "total_images": 0,
        "splits": {}
    }
    
    for split in ['train', 'val', 'test']:
        images_dir = dataset_root / split / 'images'
        labels_dir = dataset_root / split / 'labels'
        
        n_images = len(list(images_dir.glob("*.jpg"))) if images_dir.exists() else 0
        n_labels = len(list(labels_dir.glob("*.txt"))) if labels_dir.exists() else 0
        
        stats['splits'][split] = {
            "images": n_images,
            "labels": n_labels,
            "coverage": f"{n_labels/n_images*100:.1f}%" if n_images > 0 else "0%"
        }
        
        stats['total_images'] += n_images
    
    # Save to JSON
    output_path = dataset_root / 'dataset_statistics.json'
    with open(output_path, 'w') as f:
        json.dump(stats, f, indent=2)
    
    print("📊 Dataset Statistics:")
    print(json.dumps(stats, indent=2))
    print(f"\n✅ Statistics saved: {output_path}")
    
    return stats

# Generate stats
stats = generate_statistics(OUTPUT_ROOT)

## ✅ Completion Checklist

Kiểm tra tất cả bước đã hoàn thành:

In [ ]:
print("\n" + "="*60)
print("🎉 PHASE 2 PREPROCESSING COMPLETION CHECKLIST")
print("="*60)

checks = [
    ("2.1 Filter CSV", CSV_PATH.exists()),
    ("2.2 YOLO Labels", (OUTPUT_ROOT / 'all_labels').exists() and 
     len(list((OUTPUT_ROOT / 'all_labels').glob('*.txt'))) > 0),
    ("2.3 Resized Images", (OUTPUT_ROOT / 'images_resized').exists() and 
     len(list((OUTPUT_ROOT / 'images_resized').glob('*.jpg'))) > 0),
    ("2.4 Dataset Split", (OUTPUT_ROOT / 'train').exists() and 
     (OUTPUT_ROOT / 'val').exists() and 
     (OUTPUT_ROOT / 'test').exists()),
    ("2.5 data.yaml", (OUTPUT_ROOT / 'data.yaml').exists()),
    ("2.6 Visualizations", (OUTPUT_ROOT / 'visualization_samples.jpg').exists())
]

all_passed = True
for task, passed in checks:
    status = "✅" if passed else "❌"
    print(f"{status} {task}")
    if not passed:
        all_passed = False

print("="*60)
if all_passed:
    print("🎊 ALL TASKS COMPLETE! Dataset ready for training.")
else:
    print("⚠️  Some tasks incomplete. Please review above.")
print("="*60)

print(f"\n📦 Deliverables:")
print(f"  ✅ Train set: {len(list((OUTPUT_ROOT/'train'/'images').glob('*.jpg')))} images")
print(f"  ✅ Val set: {len(list((OUTPUT_ROOT/'val'/'images').glob('*.jpg')))} images")
print(f"  ✅ Test set: {len(list((OUTPUT_ROOT/'test'/'images').glob('*.jpg')))} images")
print(f"  ✅ data.yaml: {OUTPUT_ROOT / 'data.yaml'}")
print(f"\n🚀 Next Step: Handoff to Member 2 for YOLOv8-Pose training!")

## 📥 Download Dataset (Optional)

Nếu muốn download về local để chạy training offline:

In [ ]:
# Zip dataset for download
import zipfile

zip_path = OUTPUT_ROOT.parent / 'cv_dataset_ready.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for split in ['train', 'val', 'test']:
        for file_type in ['images', 'labels']:
            folder = OUTPUT_ROOT / split / file_type
            for file_path in folder.glob('*'):
                arcname = f"cv_dataset/{split}/{file_type}/{file_path.name}"
                zipf.write(file_path, arcname)
    
    # Add data.yaml
    zipf.write(OUTPUT_ROOT / 'data.yaml', 'cv_dataset/data.yaml')

print(f"✅ Dataset zipped: {zip_path}")
print(f"  Size: {zip_path.stat().st_size / 1024**2:.1f} MB")
print(f"\n📥 Download the zip file from Google Drive to use locally.")

---

## 🎯 Summary

Bạn đã hoàn thành preprocessing pipeline:

1. ✅ Filtered CSV by existing images
2. ✅ Converted annotations to YOLO pose format
3. ✅ Resized all images to 640×640
4. ✅ Split dataset (70/15/15)
5. ✅ Created data.yaml config
6. ✅ Verified and visualized samples

**Dataset is ready for YOLOv8-Pose training!**

---

### 📚 References

- [YOLOv8-Pose Documentation](https://docs.ultralytics.com/tasks/pose/)
- [YOLO Format Specification](https://docs.ultralytics.com/datasets/pose/)

---

**Created by:** AI Assistant  
**Date:** 2026-04-02  
**For:** Member 1 (CV Data Engineer)